In [1]:
# Configuración inicial
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import os
import time

# Configurar el modelo
llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    temperature=0.7  # Balance entre creatividad y consistencia
)

print("✓ Modelo configurado para zero-shot prompting")

✓ Modelo configurado para zero-shot prompting


In [4]:
# Comparación de prompts — Contexto Invermar Remuneraciones
def comparar_prompts_invermar():
    
    # Texto de ejemplo: consulta típica de un trabajador
    consulta = "Llevo 8 años en la empresa, me dijeron que me van a despedir y quiero saber cuánto me corresponde de indemnización y si tengo derecho a algo más."
    
    # ── Prompt básico (menos efectivo) ────────────────────────
    prompt_basico = f"Analiza esta consulta: {consulta}"
    
    # ── Prompt mejorado (más efectivo) ────────────────────────
    prompt_mejorado = f"""Eres un asistente especializado en remuneraciones y derecho laboral chileno de Invermar S.A.

Tarea: Analiza la siguiente consulta de un trabajador y entrega una respuesta estructurada.

Formato de respuesta:
- Tema principal: [identificar de qué trata la consulta]
- Derechos que aplican: [lista con base legal si corresponde]
- Información que falta para responder con exactitud: [lista]
- Recomendación: [qué debería hacer el trabajador]
- Derivar a Remuneraciones: [Sí/No y por qué]

Consulta del trabajador: "{consulta}"
"""
    
    print("=== COMPARACIÓN DE PROMPTS — INVERMAR ===")
    
    # ── Probar prompt básico ───────────────────────────────────
    print("\n1. PROMPT BÁSICO:")
    print(f"Prompt: {prompt_basico}")
    try:
        response1 = llm.invoke([HumanMessage(content=prompt_basico)])
        print(f"Respuesta: {response1.content}")
    except Exception as e:
        print(f"Error: {e}")
    
    print("\n" + "-" * 60)
    
    # ── Probar prompt mejorado ─────────────────────────────────
    print("\n2. PROMPT MEJORADO:")
    print(f"Prompt (primeros 120 caracteres): {prompt_mejorado[:120]}...")
    try:
        response2 = llm.invoke([HumanMessage(content=prompt_mejorado)])
        print(f"Respuesta: {response2.content}")
    except Exception as e:
        print(f"Error: {e}")
    
    print("\n=== ANÁLISIS ===")
    print("• Prompt básico   : Ambiguo, el modelo no sabe qué rol tomar ni qué formato usar")
    print("• Prompt mejorado : Define rol (asistente Invermar), estructura la respuesta y pide info relevante")

# Ejecutar comparación
comparar_prompts_invermar()

=== COMPARACIÓN DE PROMPTS — INVERMAR ===

1. PROMPT BÁSICO:
Prompt: Analiza esta consulta: Llevo 8 años en la empresa, me dijeron que me van a despedir y quiero saber cuánto me corresponde de indemnización y si tengo derecho a algo más.
Respuesta: La consulta que planteas implica analizar varios aspectos relacionados con el despido, los derechos laborales y el cálculo de la indemnización. Para responder adecuadamente, se deben tener en cuenta factores como las leyes laborales del país en el que se encuentra el trabajador, el tipo de contrato, las causas del despido y otros beneficios legales. A continuación, desglosaré los puntos clave:

---

### **1. Antigüedad en la empresa**
Mencionas que llevas **8 años trabajando en la empresa**, lo cual es un dato crucial para calcular la indemnización, ya que en muchas legislaciones laborales la antigüedad es uno de los principales factores.

- Normalmente, el cálculo de la indemnización se basa en una cantidad fija por año trabajado, que puede

In [5]:
# 1. CLASIFICACIÓN — Consultas de trabajadores Invermar
def zero_shot_clasificacion():
    print("=== ZERO-SHOT PARA CLASIFICACIÓN — INVERMAR ===")
    
    consultas = [
        "¿Cuándo me depositan el sueldo de este mes?",
        "Necesito un certificado de antigüedad para el banco",
        "Me descontaron algo que no reconozco en mi liquidación",
        "¿Cuántos días de vacaciones me quedan disponibles?",
        "Quiero saber si me corresponde bono de escolaridad"
    ]
    
    # Prompt optimizado para clasificación
    clasificacion_prompt = """Eres un clasificador de consultas laborales de Invermar S.A.

Clasifica cada consulta en una de estas categorías:
- LIQUIDACION: Dudas sobre el detalle del sueldo, descuentos o haberes
- VACACIONES: Consultas sobre días de descanso o feriados
- BENEFICIOS: Preguntas sobre bonos, aguinaldos, CCAF u otros beneficios
- DOCUMENTOS: Solicitudes de certificados, finiquitos u otros documentos
- OTRO: Consultas que no entran en las categorías anteriores

Responde solo con el nombre de la categoría.

Consulta: "{}"
Categoría:"""
    
    for i, consulta in enumerate(consultas, 1):
        prompt = clasificacion_prompt.format(consulta)
        try:
            response = llm.invoke([HumanMessage(content=prompt)])
            print(f"{i}. Consulta : {consulta}")
            print(f"   Categoría: {response.content.strip()}\n")
        except Exception as e:
            print(f"Error en consulta {i}: {e}")

# Ejecutar clasificación
zero_shot_clasificacion()

=== ZERO-SHOT PARA CLASIFICACIÓN — INVERMAR ===
1. Consulta : ¿Cuándo me depositan el sueldo de este mes?
   Categoría: OTRO

2. Consulta : Necesito un certificado de antigüedad para el banco
   Categoría: DOCUMENTOS

3. Consulta : Me descontaron algo que no reconozco en mi liquidación
   Categoría: LIQUIDACION

4. Consulta : ¿Cuántos días de vacaciones me quedan disponibles?
   Categoría: VACACIONES

5. Consulta : Quiero saber si me corresponde bono de escolaridad
   Categoría: BENEFICIOS



In [6]:
# 2. GENERACIÓN DE CONTENIDO — Comunicados Internos Invermar
def zero_shot_generacion():
    print("=== ZERO-SHOT PARA GENERACIÓN — INVERMAR ===")
    
    # Prompt estructurado para generación
    generacion_prompt = """Eres un redactor de comunicados internos de Invermar S.A., empresa salmonera chilena.

Tarea: Redactar un comunicado interno para los trabajadores sobre el tema dado.

Requisitos:
- Tono formal pero cercano
- Máximo 150 palabras
- Mencionar que pueden consultar al área de Remuneraciones
- Incluir un llamado a la acción claro

Tema: "Recordatorio de fecha de pago de remuneraciones y cómo revisar la liquidación en BUK"

Comunicado:"""
    
    try:
        response = llm.invoke([HumanMessage(content=generacion_prompt)])
        print("Comunicado generado:")
        print("-" * 40)
        print(response.content)
        print("-" * 40)
        
        # Análisis del resultado
        words        = len(response.content.split())
        menciona_buk = "BUK" in response.content or "buk" in response.content.lower()
        llamado      = "?" in response.content or any(
                           p in response.content.lower() 
                           for p in ["contáctenos", "consulte", "escríbanos", "acércate", "comuníquese"]
                       )
        
        print(f"\nAnálisis:")
        print(f"• Palabras       : {words} (límite: 150)")
        print(f"• Menciona BUK   : {'✓' if menciona_buk else '✗'}")
        print(f"• Llamado acción : {'✓' if llamado else '✗'}")
        print(f"• Cumple todo    : {'✓' if words <= 150 and menciona_buk and llamado else '✗'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar generación
zero_shot_generacion()

=== ZERO-SHOT PARA GENERACIÓN — INVERMAR ===
Comunicado generado:
----------------------------------------
**Comunicado Interno**  
*Recordatorio de fecha de pago de remuneraciones y revisión de liquidaciones en BUK*  

Estimados colaboradores,  

Les recordamos que el próximo **[día y mes]** se realizará el pago de las remuneraciones correspondientes al período en curso. Para su comodidad, pueden revisar su liquidación de sueldo directamente en la plataforma **BUK**, accediendo con su usuario y contraseña.  

Es importante que revisen detalladamente la información en su liquidación y, ante cualquier consulta o inquietud, no duden en contactar al área de **Remuneraciones**, quienes están disponibles para apoyarlos en lo que necesiten.  

Los invitamos a ingresar cuanto antes a BUK para confirmar que su información esté actualizada y resolver cualquier duda con antelación. Su compromiso y atención a este proceso son fundamentales para un correcto manejo de nuestras gestiones internas.  

In [8]:
# 3. EXTRACCIÓN DE INFORMACIÓN — Liquidaciones Invermar
def zero_shot_extraccion():
    print("=== ZERO-SHOT PARA EXTRACCIÓN — INVERMAR ===")
    
    # Texto de ejemplo: consulta real de un trabajador
    consulta_trabajador = """Hola, soy Juan Pérez, trabajo en Astilleros Calbuco hace 6 años 
    como operador de maquinaria. Me llegó mi liquidación de marzo y me descontaron 45.000 pesos 
    que no reconozco, además el bono de producción no aparece y debería ser de 80.000 pesos 
    aproximadamente. Mi sueldo base es 650.000 pesos. Necesito que me ayuden a revisar esto 
    antes del viernes porque tengo que pagar el arriendo."""
    
    # Prompt para extracción estructurada
    extraccion_prompt = f"""Eres un asistente de remuneraciones de Invermar S.A. experto en extraer datos clave de consultas laborales.

Extrae la siguiente información de la consulta del trabajador:

Formato de respuesta (JSON):
{{
  "nombre_trabajador": "nombre completo si se menciona",
  "empresa": "empresa del grupo Invermar si se menciona",
  "antiguedad": "años de servicio si se mencionan",
  "cargo": "cargo o función si se menciona",
  "sueldo_base": "monto en pesos si se menciona",
  "problema_principal": "descripción breve del problema",
  "montos_en_disputa": ["lista de montos mencionados con contexto"],
  "urgencia": "Alta / Media / Baja según el mensaje"
}}

Consulta: "{consulta_trabajador}"

JSON:"""
    
    try:
        response = llm.invoke([HumanMessage(content=extraccion_prompt)])
        print("Información extraída:")
        print(response.content)
        
        # Intentar parsear como JSON para validar formato
        import json
        try:
            # Limpiar posibles ```json ... ``` que el modelo a veces agrega
            texto_limpio = response.content.strip().removeprefix("```json").removesuffix("```").strip()
            data = json.loads(texto_limpio)
            print("\n✓ Formato JSON válido")
            print(f"✓ Campos extraídos : {len(data)}")
            print(f"✓ Urgencia detectada: {data.get('urgencia', 'No detectada')}")
            print(f"✓ Problema principal: {data.get('problema_principal', '-')}")
        except:
            print("\n✗ Formato JSON inválido — necesita refinamiento del prompt")
            
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar extracción

In [9]:
# Técnica 1: Prompts con Roles Específicos — Invermar Remuneraciones
def roles_especificos():
    print("=== OPTIMIZACIÓN: ROLES ESPECÍFICOS — INVERMAR ===")
    
    pregunta = "Un trabajador lleva 5 años en la empresa y fue despedido sin causa justificada. ¿Qué le corresponde?"
    
    roles = {
        "Genérico": "Responde la siguiente pregunta:",
        "Asistente de Remuneraciones": "Eres un asistente de remuneraciones de Invermar S.A. Responde de forma clara y amable para un trabajador:",
        "Experto en Derecho Laboral": "Eres un abogado laboralista chileno con 10 años de experiencia en el Código del Trabajo. Responde con precisión legal citando artículos:",
        "Jefe de RRHH": "Eres el Jefe de Recursos Humanos de una empresa salmonera. Responde considerando tanto los derechos del trabajador como los procesos internos de la empresa:"
    }
    
    for rol, contexto in roles.items():
        prompt = f"{contexto}\n\nPregunta: {pregunta}"
        
        print(f"\n{rol.upper()}:")
        print("-" * 30)
        
        try:
            response = llm.invoke([HumanMessage(content=prompt)])
            # Mostrar solo los primeros 200 caracteres para comparar enfoque
            preview = response.content[:200] + "..." if len(response.content) > 200 else response.content
            print(preview)
        except Exception as e:
            print(f"Error: {e}")
    
    print("\n=== OBSERVACIONES ===")
    print("• Rol genérico      : Respuesta vaga, sin base legal ni contexto laboral")
    print("• Asistente RRHH    : Tono amable, orientado al trabajador, lenguaje simple")
    print("• Experto laboral   : Cita artículos del CT, más preciso pero más técnico")
    print("• Jefe de RRHH      : Balancea derechos del trabajador con procesos internos")

# Ejecutar ejemplo de roles
roles_especificos()

=== OPTIMIZACIÓN: ROLES ESPECÍFICOS — INVERMAR ===

GENÉRICO:
------------------------------
Cuando un trabajador lleva 5 años en una empresa y es despedido sin causa justificada, le corresponden ciertos derechos y beneficios, dependiendo del país en el que ocurra el despido, ya que las leyes...

ASISTENTE DE REMUNERACIONES:
------------------------------
¡Hola! Lamento mucho que estés pasando por esta situación. Permíteme explicarte de manera clara lo que te corresponde en caso de un despido sin causa justificada, considerando que llevas 5 años trabaj...

EXPERTO EN DERECHO LABORAL:
------------------------------
¡Entendido! En el caso de un trabajador chileno con 5 años de antigüedad en una empresa que fue despedido sin causa justificada, se deben analizar los artículos pertinentes del **Código del Trabajo** ...

JEFE DE RRHH:
------------------------------
Como jefe de Recursos Humanos de una empresa salmonera, es fundamental garantizar que nuestras acciones se alineen con la legisl

In [10]:
# Técnica 2: Instrucciones Graduales — Invermar Remuneraciones
def instrucciones_graduales():
    print("=== OPTIMIZACIÓN: INSTRUCCIONES GRADUALES — INVERMAR ===")
    
    # Caso real de finiquito
    caso = "Un trabajador de Pesquera La Portada lleva 7 años en la empresa, con sueldo base de $650.000 y un promedio de bonos variables de $120.000 en los últimos 3 meses. Fue despedido por necesidades de la empresa (Art. 161)."
    
    # Prompt con instrucciones graduales
    prompt_gradual = f"""Eres un experto en remuneraciones y derecho laboral chileno de Invermar S.A.

Caso: {caso}

Calcula el finiquito siguiendo este proceso paso a paso:

PASO 1: Base de cálculo
- Determina la remuneración mensual del Art. 172 (sueldo base + promedio variable)
- Indica qué conceptos se incluyen y cuáles no

PASO 2: Indemnización por años de servicio
- Calcula el monto según Art. 163 del Código del Trabajo
- Indica el tope legal aplicable

PASO 3: Vacaciones proporcionales
- Estima los días proporcionales pendientes
- Calcula el monto correspondiente

PASO 4: Otros conceptos del finiquito
- Lista otros ítems que podrían corresponder (feriado proporcional, meses trabajados, etc.)
- Indica cuáles dependen de información adicional

Desarrolla cada paso con detalle:"""
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_gradual)])
        print("FINIQUITO DESARROLLADO:")
        print("=" * 50)
        print(response.content)
        
        # Análisis de estructura
        pasos_detectados    = sum(1 for i in range(1, 5) if f"PASO {i}" in response.content)
        secciones           = ["art. 172", "art. 163", "vacacion", "indemnizacion"]
        secciones_encontradas = sum(1 for s in secciones if s.lower() in response.content.lower())
        
        print("\n=== ANÁLISIS DE ESTRUCTURA ===")
        print(f"• Pasos identificados  : {pasos_detectados}/4")
        print(f"• Conceptos laborales  : {secciones_encontradas}/4")
        print(f"• Estructura seguida   : {'✓' if pasos_detectados >= 3 else '✗'}")
        print(f"• Listo para validar   : {'✓' if pasos_detectados == 4 and secciones_encontradas >= 3 else '✗ — revisar prompt'}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar instrucciones graduales
instrucciones_graduales()

=== OPTIMIZACIÓN: INSTRUCCIONES GRADUALES — INVERMAR ===
FINIQUITO DESARROLLADO:
¡Claro! A continuación, se detalla el cálculo del finiquito para el trabajador de Pesquera La Portada conforme al derecho laboral chileno vigente y a las condiciones descritas:

---

### **PASO 1: Base de cálculo**
La base de cálculo para el finiquito está regulada por el **Art. 172 del Código del Trabajo**, que establece que la remuneración mensual es la suma del sueldo base más las asignaciones de carácter fijo y los promedios de las remuneraciones variables de los últimos 3 meses trabajados, excluyendo ciertos conceptos.

#### **Remuneración mensual del Art. 172**
1. **Sueldo base**: $650.000
2. **Promedio de bonos variables**: $120.000  
   (Se indica que corresponde al promedio de los últimos 3 meses, por lo que se incluye).
3. **Total base de cálculo**:  
   **$650.000 + $120.000 = $770.000**

#### **Conceptos incluidos en la base de cálculo**
- Sueldo base.
- Bonos y gratificaciones variables, **pro

In [12]:
# Comparación Zero-Shot vs Few-Shot — Invermar Remuneraciones
def comparar_zero_vs_few_shot():
    print("=== COMPARACIÓN: ZERO-SHOT vs FEW-SHOT — INVERMAR ===")
    
    # Consulta nueva a clasificar
    nueva_consulta = "Me dijeron que este mes no me pagaron el bono de producción y no sé por qué."
    
    # ── Enfoque Zero-Shot ──────────────────────────────────────
    prompt_zero_shot = f"""Clasifica esta consulta laboral en: LIQUIDACION, VACACIONES, BENEFICIOS, DOCUMENTOS u OTRO:

Consulta: "{nueva_consulta}"
Categoría:"""
    
    # ── Enfoque Few-Shot ───────────────────────────────────────
    prompt_few_shot = f"""Clasifica cada consulta laboral en: LIQUIDACION, VACACIONES, BENEFICIOS, DOCUMENTOS u OTRO:

Consulta: "No entiendo por qué me descontaron más de lo normal este mes."
Categoría: LIQUIDACION

Consulta: "¿Cuántos días de vacaciones me quedan disponibles?"
Categoría: VACACIONES

Consulta: "Necesito un certificado de antigüedad para el banco."
Categoría: DOCUMENTOS

Consulta: "¿Me corresponde el bono de escolaridad si mi contrato es a plazo fijo?"
Categoría: BENEFICIOS

Consulta: "{nueva_consulta}"
Categoría:"""
    
    # ── Probar ambos enfoques ──────────────────────────────────
    print("\n1. ZERO-SHOT:")
    print("-" * 15)
    try:
        response_zero = llm.invoke([HumanMessage(content=prompt_zero_shot)])
        print(f"Resultado: {response_zero.content.strip()}")
    except Exception as e:
        print(f"Error: {e}")
    
    print("\n2. FEW-SHOT:")
    print("-" * 15)
    try:
        response_few = llm.invoke([HumanMessage(content=prompt_few_shot)])
        print(f"Resultado: {response_few.content.strip()}")
    except Exception as e:
        print(f"Error: {e}")
    
    print("\n=== ANÁLISIS ===")
    print("• Zero-shot : Puede devolver texto extra o categoría incorrecta")
    print("• Few-shot  : Sigue el patrón exacto y es más consistente")
    print("• Few-shot  : Ideal para el clasificador del bot de Invermar")
    
    # Comparación de tokens
    tokens_zero = len(prompt_zero_shot.split())
    tokens_few  = len(prompt_few_shot.split())
    print(f"\n• Tokens zero-shot : ~{tokens_zero}")
    print(f"• Tokens few-shot  : ~{tokens_few} ({tokens_few/tokens_zero:.1f}x más)")
    print(f"• Trade-off        : Few-shot usa más tokens pero clasifica mejor")

# Ejecutar comparación
comparar_zero_vs_few_shot()

=== COMPARACIÓN: ZERO-SHOT vs FEW-SHOT — INVERMAR ===

1. ZERO-SHOT:
---------------
Resultado: BENEFICIOS

2. FEW-SHOT:
---------------
Resultado: CATEGORÍA: BENEFICIOS

=== ANÁLISIS ===
• Zero-shot : Puede devolver texto extra o categoría incorrecta
• Few-shot  : Sigue el patrón exacto y es más consistente
• Few-shot  : Ideal para el clasificador del bot de Invermar

• Tokens zero-shot : ~30
• Tokens few-shot  : ~82 (2.7x más)
• Trade-off        : Few-shot usa más tokens pero clasifica mejor
